In [ ]:
import chess
import json
import numpy as np
import torch
import os
from tqdm import tqdm
from decimal import Decimal
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import defaultdict

RED = '\033[31m'
GREEN = '\033[32m'
BLUE = '\033[34m'
RESET = '\033[0m'

In [ ]:
def generate_templates(board, move, san) -> list[str]:
    piece = board.piece_at(move.from_square)
    target = chess.square_name(move.to_square)

    is_capture = board.is_capture(move)
    is_castle = board.is_castling(move)
    is_promotion = move.promotion is not None

    templates = []

    file_name = chess.FILE_NAMES[chess.square_file(move.from_square)]
    
    if is_castle:
        if chess.square_file(move.to_square) == 6:
            templates += [
                "Castle kingside",
                "Kingside castle",
                "Perform kingside castling",
                "King castles short",
                "Castle short",
                "Short castle", 
            ]
        else:
            templates += [
                "Castle queenside",
                "Queenside castle",
                "Perform queenside castling",
                "King castles long",
                "Castle long",
                "Long castle",
            ]
        return templates

    if is_promotion:
        promo_piece = chess.piece_name(move.promotion)
        templates += [
            f"Promote pawn to {promo_piece} on {target}",
            f"Advance pawn to {target} and promote to {promo_piece}",
            f"Move pawn to {target} and make it a {promo_piece}",
        ]
        return templates

    p_name = chess.piece_name(piece.piece_type)

    if is_capture:
        templates += [
            f"{p_name.capitalize()} captures on {target}",
            f"Capture the piece on {target} with the {p_name}",
            f"Take on {target} using the {p_name}",
            f"Use the {p_name} to capture on {target}",
        ]
    elif piece.piece_type == chess.PAWN:
        templates += [
            f"Advance pawn to {target}",
            f"Push the pawn to {target}",
            f"Move pawn to {target}",
            f"Advance {file_name} pawn",
        ]
    else:
        templates += [
            f"Move the {p_name} to {target}",
            f"Play {p_name} to {target}",
            f"Develop the {p_name} to {target}",
            f"{p_name.capitalize()} goes to {target}",
        ]

    templates.append(f"Play {san}")

    return templates

In [ ]:
def get_move_descriptions(board) -> list[dict]:
    dataset = []
    for move in board.legal_moves:
        san = board.san(move)
        for template in generate_templates(board, move, san):
            dataset.append(
                {
                    "move": san,
                    "desc": template
                }
            )
    return dataset

In [ ]:
if __name__ == "__main__":
    EMBEDDING_MODELS = {
        "MiniLM-L6":   "all-MiniLM-L6-v2",
        "mpnet-base":  "all-mpnet-base-v2",
        "bge-small":   "BAAI/bge-small-en-v1.5",
    }
    
    loaded_models = {}
    for short_name, model_id in EMBEDDING_MODELS.items():
        print(f"Loading {model_id}...")
        loaded_models[short_name] = SentenceTransformer(model_id)
    print(f"All {len(loaded_models)} embedding models loaded ✓")
    
    model = loaded_models["MiniLM-L6"]

In [ ]:
def build_index(records: list[dict], embed_model=None) -> tuple[list, np.ndarray]:
    """
    Embeds all descriptions and returns:
      - the original records list  (so you can look up move_san by index)
      - a matrix of embeddings    (shape: N x embedding_dim)
    """
    if embed_model is None:
        embed_model = model
    descriptions = [r["desc"] for r in records]
    embeddings = embed_model.encode(descriptions, convert_to_numpy=True)
    return records, embeddings

In [ ]:
def find_best_move(user_query: str, records: list[dict], embeddings: np.ndarray, embed_model=None) -> str:
    if embed_model is None:
        embed_model = model
    query_vec = embed_model.encode(user_query, convert_to_numpy=True)
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_vec)
    scores = embeddings @ query_vec / norms
    best_idx = int(np.argmax(scores))
    return records[best_idx]["move"]

In [ ]:
def process_board_state(board: chess.Board, user_query: str) -> str:
    """
    All-in-one function Thomas can call in his game loop:
      board       – the current chess.Board object
      user_query  – whatever the user typed

    Returns the matched SAN move string (e.g. "Nf3", "O-O", "exd5").
    """
    records = get_move_descriptions(board)
    records, embeddings = build_index(records)
    return find_best_move(user_query, records, embeddings)

In [ ]:
if __name__ == "__main__":
    # ── Test: simulate one turn ──────────────────────────────────────────────────
    test_board = chess.Board()  # standard starting position
    
    test_queries = [
        "push the pawn in the center",
        "bring out the knight on the right",
        "move a pawn forward",
        "develop a piece",
    ]
    
    for query in test_queries:
        matched = process_board_state(test_board, query)
        print(f"Query:  '{query}'")
        print(f"Matched: {matched}")
        print()

In [ ]:
if __name__ == "__main__":
    LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

    print(f"Loading {LLM_MODEL_ID} tokenizer + model (this may take a minute)...")
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    print(f"{LLM_MODEL_ID} loaded ✓")
    
    SMOL_MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    
    print(f"Loading {SMOL_MODEL_ID} tokenizer + model...")
    smol_tokenizer = AutoTokenizer.from_pretrained(SMOL_MODEL_ID)
    # Use float32 — if the GPU is full from Qwen, device_map="auto" places this
    # model on CPU where float16 silently produces garbage / repeated tokens.
    smol_model = AutoModelForCausalLM.from_pretrained(
        SMOL_MODEL_ID,
        torch_dtype=torch.float32,
        device_map="auto",
    )
    print(f"{SMOL_MODEL_ID} loaded ✓  (device: {next(smol_model.parameters()).device})")

In [ ]:
def _llm_pick_move_with(tokenizer, model, user_query: str, board: chess.Board) -> str:
    """
    Generic helper: ask any causal LLM to choose the best SAN move.
    Returns a SAN string (or '' if parsing fails).
    """
    legal_sans = sorted([board.san(m) for m in board.legal_moves])
    fen = board.fen()

    prompt = (
        f"You are a chess assistant. Given the board position (FEN) and a list of legal moves, "
        f"pick the single best move that matches the user's intent.\n\n"
        f"FEN: {fen}\n"
        f"Legal moves: {', '.join(legal_sans)}\n"
        f"User says: \"{user_query}\"\n\n"
        f"Reply with ONLY the move in standard algebraic notation (SAN), nothing else."
    )

    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    cleaned = raw.split()[0] if raw else ""
    if cleaned in legal_sans:
        return cleaned
    for san in legal_sans:
        if san in raw:
            return san
    return cleaned

In [ ]:
def llm_pick_move(user_query: str, board: chess.Board) -> str:
    """Qwen 2.5 1.5B wrapper."""
    return _llm_pick_move_with(llm_tokenizer, llm_model, user_query, board)

In [ ]:
def smol_pick_move(user_query: str, board: chess.Board) -> str:
    """SmolLM2 1.7B wrapper."""
    return _llm_pick_move_with(smol_tokenizer, smol_model, user_query, board)

In [ ]:
if __name__ == "__main__":
    _board = chess.Board()
    _test = llm_pick_move("push the king's pawn forward two squares", _board)
    print(f"Qwen picked: {_test}")
    
    _test2 = smol_pick_move("push the king's pawn forward two squares", _board)
    print(f"SmolLM2 picked: {_test2}")

In [ ]:
if __name__ == "__main__":
    with open("testing.json", "r") as f:
        TEST_CASES = json.load(f)
        print(TEST_CASES)

In [ ]:
def claude_pick_move(user_query: str, board: chess.Board) -> str:
    """Claude Sonnet 3.5 API wrapper using Anthropic"""
    try:
        import anthropic
    except ImportError:
        return "N/A (install anthropic)"

    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return "N/A (no API key)"

    client = anthropic.Anthropic(api_key=api_key)
    legal_sans = sorted([board.san(m) for m in board.legal_moves])
    fen = board.fen()
    
    prompt = (
        f"You are a chess assistant. Given the board position (FEN) and a list of legal moves, "
        f"pick the single best move that matches the user's intent.\n\n"
        f"FEN: {fen}\n"
        f"Legal moves: {', '.join(legal_sans)}\n"
        f"User says: \"{user_query}\"\n\n"
        f"Reply with ONLY the move in standard algebraic notation (SAN), nothing else. Do not add punctuation."
    )
    
    try:
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=16,
            temperature=0,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = response.content[0].text.strip()
        cleaned = raw.split()[0] if raw else ""
        if cleaned in legal_sans:
            return cleaned
        for san in legal_sans:
            if san in raw:
                return san
        return cleaned
    except Exception as e:
        return f"Error: {e}"

In [ ]:
if __name__ == "__main__":
    LLM_MODELS = {
        "Claude Sonnet 3.5": claude_pick_move,
        "Qwen 1.5B":  llm_pick_move,
        "SmolLM2 1.7B": smol_pick_move,
    }

In [ ]:
def find_top_k(user_query, records, embeddings, embed_model, k=3):
    query_vec = embed_model.encode(user_query, convert_to_numpy=True)
    norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_vec)
    scores = embeddings @ query_vec / (norms + 1e-9)
    top_indices = np.argsort(scores)[::-1]
    seen, results = set(), []
    for idx in top_indices:
        san = records[idx]["move"]
        if san not in seen:
            seen.add(san)
            results.append((san, float(scores[idx])))
        if len(results) >= k:
            break
    return results

In [ ]:
if __name__ == "__main__":
    boards_and_records = {}
    test_id = 0
    for datapoint in TEST_CASES:
        move_seq = datapoint["prev"].replace('[', '').replace(']', '').replace("'", '').split(', ')
        if len(move_seq) > 0 and move_seq[0] == '':
            move_seq = []
        query = datapoint["desc"]
        expected = datapoint["move"]
        fen_override = datapoint["fen"]
        test_id += 1
        if fen_override != "None":
            board = chess.Board(fen_override)
        else:
            board = chess.Board()
            for mv in move_seq:
                board.push_san(mv)
        legal_sans = [board.san(m) for m in board.legal_moves]
        if expected not in legal_sans:
            boards_and_records[test_id] = None
            continue
        records = get_move_descriptions(board)
        boards_and_records[test_id] = (board, records, legal_sans)
    
    embed_names = list(loaded_models.keys())
    llm_names   = list(LLM_MODELS.keys())
    col_w = 14
    header_models = "".join(f"{n:<{col_w}}" for n in embed_names)
    header_llms   = "".join(f"{n:<{col_w}}" for n in llm_names)
    
    print("=" * 140)
    print(f"  CHESS NL→SAN EVALUATION  |  {len(embed_names)} Embedding Models + {len(llm_names)} LLMs  |  {len(TEST_CASES)} tests")
    print("=" * 140)
    print(f"  {'#':<4} {'Query':<36} {header_models}{header_llms}{'Exp':<8}")
    print("-" * 140)

    scores_top1 = defaultdict(int)
    scores_top3 = defaultdict(int)
    llm_correct = defaultdict(int)
    total = 0
    all_results = []
    
    test_id = 0
    for datapoint in TEST_CASES:
        move_seq = datapoint["prev"].replace('[', '').replace(']', '').replace("'", '').split(', ')
        if len(move_seq) > 0 and move_seq[0] == '':
            move_seq = []
        query = datapoint["desc"]
        expected = datapoint["move"]
        fen_override = datapoint["fen"]
        test_id += 1
        
        entry = boards_and_records[test_id]
        if entry is None:
            print(f"  {test_id:<4} [SKIP – expected move not legal]")
            continue
    
        board, records, legal_sans = entry
        total += 1
    
        row_preds = {}
        row_ok    = {}
    
        # ── Each embedding model ──
        for name, emb_model in loaded_models.items():
            recs, embs = build_index(records, embed_model=emb_model)
            top3 = find_top_k(query, recs, embs, emb_model, k=3)
            top3_sans = [s for s, _ in top3]
            pred = top3_sans[0] if top3_sans else "—"
            ok = pred == expected
            t3 = expected in top3_sans
            if ok: scores_top1[name] += 1
            if t3: scores_top3[name] += 1
            row_preds[name] = pred
            row_ok[name] = ok
    
        # ── Each LLM ──
        for llm_name, llm_fn in LLM_MODELS.items():
            pred = llm_fn(query, board)
            ok = pred == expected
            if ok: llm_correct[llm_name] += 1
            row_preds[llm_name] = pred
            row_ok[llm_name] = ok
    
        # ── Print row ──
        q_display = (query[:33] + "…") if len(query) > 34 else query
        cells = ""
        # ✅
        for name in embed_names:
            if row_ok[name]:
                mark = f"{GREEN}✓{RESET}"
                cells += f"{GREEN}{row_preds[name]:<11}{mark}  {RESET}"
            else:
                mark = f"{RED}✗{RESET}"
                cells += f"{RED}{row_preds[name]:<11}{mark}  {RESET}"
        for llm_name in llm_names:
            if row_ok[llm_name]:
                mark = f"{GREEN}✓{RESET}"
                cells += f"{GREEN}{row_preds[llm_name]:<11}{mark}  {RESET}"
            else:
                mark = f"{RED}✗{RESET}"
                cells += f"{RED}{row_preds[llm_name]:<11}{mark}  {RESET}"
        print(f"  {test_id:<4} {q_display:<36} {cells}{expected}")
    
        all_results.append({
            "id": test_id, "query": query, "expected": expected,
            "preds": {**row_preds},
            "correct": {**row_ok},
        })
    
    # ── Summary ───────────────────────────────────────────────────────────────────
    print("=" * 140)
    print(f"\n  RESULTS SUMMARY  (n={total})\n")
    print(f"  {'Model':<20} {'Top-1':>8} {'Top-3':>8} {'Top-1 %':>10}")
    print(f"  {'-'*20} {'-'*8} {'-'*8} {'-'*10}")
    for name in embed_names:
        t1 = scores_top1[name]
        t3 = scores_top3[name]
        print(f"  {name:<20} {t1:>5}/{total}  {t3:>5}/{total}  {t1/total*100:>8.1f}%")
    for llm_name in llm_names:
        lc = llm_correct[llm_name]
        print(f"  {llm_name:<20} {lc:>5}/{total}  {'—':>8}  {lc/total*100:>8.1f}%")
    print()
    
    # ── Best embedding vs best LLM ───────────────────────────────────────────────
    best_embed_name = max(embed_names, key=lambda n: scores_top1[n])
    best_embed_acc  = scores_top1[best_embed_name] / total * 100
    best_llm_name   = max(llm_names, key=lambda n: llm_correct[n])
    best_llm_acc    = llm_correct[best_llm_name] / total * 100
    print(f"  Best embedding: {best_embed_name} ({best_embed_acc:.1f}%)  vs  Best LLM: {best_llm_name} ({best_llm_acc:.1f}%)  "
          f"Δ = {best_embed_acc - best_llm_acc:+.1f} pp")
    
    # ── Per-model misses ──────────────────────────────────────────────────────────
    print()
    for name in embed_names + llm_names:
        misses = [r for r in all_results if not r["correct"][name]]
        if misses:
            print(f"  {name} misses:")
            for r in misses:
                print(f"    #{r['id']:>2}  Got '{r['preds'][name]}' expected '{r['expected']}'")
        else:
            print(f"{GREEN}  {name}: perfect ✓{RESET}")
    
    print("=" * 140)